In [1]:
pip install scikit-rf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 631.3/631.3 kB 8.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import skrf as rf
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import os
import skrf as rf

In [3]:
def plotter(file_path, frequency, magnitude, valley_f, valley_y, bw=None):


    plt.figure()
    plt.plot(frequency, magnitude, label="S21 (dB)")
    plt.plot(valley_f, valley_y, "rv", label="Resonance (valley)")

    if bw is not None:
        fL, fR, bw5, thr = bw
        # Vertical dotted lines at fL and fR
        plt.axvline(fL, linestyle="--", color="k", alpha=0.7, label="5 dB edges")
        plt.axvline(fR, linestyle="--", color="k", alpha=0.7)

        # Optional: horizontal threshold line (helps visually)
        plt.axhline(thr, linestyle=":", color="gray", alpha=0.7, label="-5 dB level")

    volunteer = file_path.split('/')[6]
    device = file_path.split('/')[7]
    plt.title(volunteer+device)
    plt.xlabel("Frequency (GHz)")
    plt.ylabel("Magnitude (dB)")
    plt.grid(True)
    plt.legend()
    plt.show()
    plt.close()

In [12]:
from pathlib import Path
import os
import matplotlib.pyplot as plt

def save_plotter(
    file_path,
    frequency,
    magnitude,
    valley_f,
    valley_y,
    bw=None,
    save_dir="/content/drive/MyDrive/Data/plots/2022"
):

    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(8, 5))

    plt.plot(frequency, magnitude, label="S21 (dB)")
    plt.plot(valley_f, valley_y, "rv", label="Resonance")

    if bw is not None:
        fL, fR, bw5, thr = bw

        plt.axvline(
            fL,
            linestyle="--",
            color="k",
            alpha=0.7,
            label="5 dB edges"
        )

        plt.axvline(
            fR,
            linestyle="--",
            color="k",
            alpha=0.7
        )

        plt.axhline(
            thr,
            linestyle=":",
            color="gray",
            alpha=0.7,
            label="-5 dB level"
        )

    path_obj = Path(file_path)

    # safer folder extraction
    try:
        volunteer = path_obj.parts[-3]
        device = path_obj.parts[-2]
    except:
        volunteer = "unknown_volunteer"
        device = "unknown_device"

    plt.title(f"{volunteer} | {device}")

    plt.xlabel("Frequency (GHz)")
    plt.ylabel("Magnitude (dB)")

    plt.grid(True)
    plt.legend()

    base_name = path_obj.stem + ".png"

    filename = f"{volunteer}_{device}_{base_name}"

    save_path = os.path.join(save_dir, filename)

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

In [5]:
from scipy.signal import find_peaks
import numpy as np

def find_bandstop(freq, s21_db, min_freq=1.75, max_freq=3.0,
                  prominence=3.0, min_depth_db=-20):
    freq = np.asarray(freq)
    s21_db = np.asarray(s21_db)

    mask = (freq >= min_freq) & (freq <= max_freq)
    f = freq[mask]
    y = s21_db[mask]

    if len(f) == 0:
        raise ValueError("No frequency samples in resonance search range")

    # valleys in S21 are peaks in -S21
    valleys, props = find_peaks(-y, prominence=prominence)

    if len(valleys) == 0:
        raise ValueError("No resonance valley found")

    candidates = []
    for idx, prom in zip(valleys, props["prominences"]):
        depth = y[idx]
        if depth <= min_depth_db:
            candidates.append((idx, f[idx], depth, prom))

    if not candidates:
        raise ValueError("No valley passed depth threshold")

    # choose deepest valley, not first valley
    best_idx, best_freq, best_depth, best_prom = min(
        candidates, key=lambda x: x[2]
    )

    return {
        "resonance_frequency": best_freq,
        "resonance_depth_db": best_depth,
        "prominence": best_prom,
        "index": best_idx,
        "num_candidates": len(candidates)
    }

In [6]:
def _interp_x_at_y(x: np.ndarray, y: np.ndarray, y0: float) -> float:
    """
    Linear interpolate x where y crosses y0.
    Assumes x is increasing and y crosses y0 in the given x,y segment.
    """
    s = y - y0
    idx = np.where(np.sign(s[:-1]) != np.sign(s[1:]))[0]
    if len(idx) == 0:
        raise ValueError("No crossing found for the given y0 in this segment.")
    i = idx[0]
    x1, x2 = x[i], x[i + 1]
    y1, y2 = y[i], y[i + 1]
    if y2 == y1:
        return float(x1)
    return float(x1 + (y0 - y1) * (x2 - x1) / (y2 - y1))

In [7]:
import numpy as np



def five_db_bandwidth(freq_ghz: np.ndarray,
                      mag_db: np.ndarray,
                      valley_freq_ghz: float,
                      valley_mag_db: float,
                      delta_db: float = 5.0):
    """
    Compute the 5 dB notch bandwidth around a bandstop resonance (valley).

    Threshold is valley_mag_db + delta_db (e.g., +5 dB above the notch minimum).
    Bandwidth is between the two frequencies where mag_db crosses this threshold.

    Returns:
      f_left, f_right, bw, threshold
    """
    freq_ghz = np.asarray(freq_ghz)
    mag_db = np.asarray(mag_db)

    # Resonance (valley) index: closest frequency sample to valley_freq_ghz
    i0 = int(np.argmin(np.abs(freq_ghz - valley_freq_ghz)))

    threshold = valley_mag_db + delta_db

    # Find left crossing: moving left from i0 until we go ABOVE threshold
    i = i0
    while i > 0 and mag_db[i] <= threshold:
        i -= 1
    if i == 0 and mag_db[i] <= threshold:
        raise ValueError("Left 5 dB crossing not found (notch never rises above threshold on left).")

    # Crossing lies between i and i+1 (because at i we are > threshold, at i+1 we are <= threshold)
    f_left = _interp_x_at_y(freq_ghz[i:i+2], mag_db[i:i+2], threshold)

    # Find right crossing: moving right from i0 until we go ABOVE threshold
    j = i0
    while j < len(freq_ghz) - 1 and mag_db[j] <= threshold:
        j += 1
    if j == len(freq_ghz) - 1 and mag_db[j] <= threshold:
        raise ValueError("Right 5 dB crossing not found (notch never rises above threshold on right).")

    # Crossing lies between j-1 and j (because at j-1 we are <= threshold, at j we are > threshold)
    f_right = _interp_x_at_y(freq_ghz[j-1:j+1], mag_db[j-1:j+1], threshold)

    bw = f_right - f_left
    return f_left, f_right, bw, threshold

In [8]:
import numpy as np
import skrf as rf

def phase_offsets_from_resonance(file_path, resonance_freq_ghz, offsets_ghz=(1.0, 2.0)):
    """
    Returns unwrapped S21 phase (degrees) at frequencies:
      resonance_freq_ghz - offset for each offset in offsets_ghz.
    """
    ntwk = rf.Network(file_path)

    freq_ghz = ntwk.f / 1e9
    s21 = ntwk.s[:, 1, 0]

    # phase in radians -> unwrap -> degrees
    phase_rad = np.angle(s21)
    phase_unwrapped_deg = np.degrees(np.unwrap(phase_rad))

    results = {}
    for off in offsets_ghz:
        target_f = resonance_freq_ghz - off

        if target_f < freq_ghz.min() or target_f > freq_ghz.max():
            results[f"phase_{off}ghz_deg"] = np.nan
            continue

        # nearest sample (simple + robust)
        idx = int(np.argmin(np.abs(freq_ghz - target_f)))
        results[f"phase_{off}ghz_deg"] = float(phase_unwrapped_deg[idx])

    return results

In [9]:
import numpy as np
import skrf as rf

def phase_slope_around_resonance(file_path, f0_ghz, window_ghz=0.1):
    """
    Slope of unwrapped S21 phase around resonance using a linear fit in [f0-window, f0+window].
    Returns slope in deg/GHz.
    """
    ntwk = rf.Network(file_path)
    freq_ghz = ntwk.f / 1e9
    s21 = ntwk.s[:, 1, 0]

    phase_deg = np.degrees(np.unwrap(np.angle(s21)))

    # select points in window around f0
    mask = (freq_ghz >= f0_ghz - window_ghz) & (freq_ghz <= f0_ghz + window_ghz)
    if mask.sum() < 3:
        return np.nan  # not enough points to fit reliably

    x = freq_ghz[mask]
    y = phase_deg[mask]

    # linear regression: y = m*x + b
    m, b = np.polyfit(x, y, 1)
    return float(m)  # deg/GHz

In [10]:
def unwrapper(file_path):
    try:
        ntwk = rf.Network(file_path)

        frequency = ntwk.f / 1e9  # GHz
        magnitude = 20 * np.log10(np.abs(ntwk.s[:, 1, 0]))  # S21 dB

        bandstop_result = find_bandstop(frequency, magnitude)

        # Supports both old tuple-style and new dict-style find_bandstop
        if isinstance(bandstop_result, dict):
            bandstop_freq = bandstop_result["resonance_frequency"]
            bandstop_mag = bandstop_result["resonance_depth_db"]
            prominence = bandstop_result.get("prominence", np.nan)
            num_candidates = bandstop_result.get("num_candidates", np.nan)
        else:
            bandstop_freq, bandstop_mag = bandstop_result
            prominence = np.nan
            num_candidates = np.nan

        bw_info = five_db_bandwidth(
            frequency,
            magnitude,
            bandstop_freq,
            bandstop_mag
        )
        f_left, f_right, bw, threshold = bw_info

        phase_feats = phase_offsets_from_resonance(
            file_path,
            bandstop_freq,
            offsets_ghz=(1.0, 2.0)
        )
        phase_1ghz_deg = phase_feats["phase_1.0ghz_deg"]
        phase_2ghz_deg = phase_feats["phase_2.0ghz_deg"]

        slope = phase_slope_around_resonance(
            file_path,
            bandstop_freq,
            window_ghz=0.1
        )

        return {
            "ok": True,
            "file_path": file_path,
            "frequency": frequency,
            "magnitude": magnitude,
            "bandstop_freq": bandstop_freq,
            "bandstop_mag": bandstop_mag,
            "f_left": f_left,
            "f_right": f_right,
            "bw": bw,
            "threshold": threshold,
            "phase_1ghz_deg": phase_1ghz_deg,
            "phase_2ghz_deg": phase_2ghz_deg,
            "slope": slope,
            "prominence": prominence,
            "num_candidates": num_candidates,
            "error": None
        }

    except Exception as e:
        return {
            "ok": False,
            "file_path": file_path,
            "frequency": None,
            "magnitude": None,
            "bandstop_freq": np.nan,
            "bandstop_mag": np.nan,
            "f_left": np.nan,
            "f_right": np.nan,
            "bw": np.nan,
            "threshold": np.nan,
            "phase_1ghz_deg": np.nan,
            "phase_2ghz_deg": np.nan,
            "slope": np.nan,
            "prominence": np.nan,
            "num_candidates": np.nan,
            "error": repr(e)
        }

In [11]:
def error_unwrapper(file_path):
  ntwk = rf.Network(file_path)

  frequency = ntwk.f / 1e9  # GHz
  magnitude = 20*np.log10(np.abs(ntwk.s[:, 1, 0]))   # S21 (dB)

  bandstop_freq, bandstop_mag = find_bandstop(frequency,magnitude)

  plotter(file_path, frequency, magnitude, bandstop_freq, bandstop_mag)



In [13]:
import os
import pandas as pd

root_dir = "/content/drive/MyDrive/Data/MAS Volunteer Study March 2022"

print(f"Traversing directory: {root_dir}")

data = []
error_data = []

for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)

        result = unwrapper(file_path)

        if not result["ok"]:
            error_data.append(result)
            continue

        data.append(result)

        save_plotter(
            result["file_path"],
            result["frequency"],
            result["magnitude"],
            result["bandstop_freq"],
            result["bandstop_mag"],
            bw=(result["f_left"],result["f_right"],result["bw"],result["threshold"])
)

features_df = pd.DataFrame(data).drop(
    columns=["ok", "frequency", "magnitude", "error"], errors='ignore'
)

error_df = pd.DataFrame(error_data)

print(f"Successful files: {len(data)}")
print(f"Failed files: {len(error_data)}")

error_df.to_csv("failed_resonance_files.csv", index=False)
features_df.to_csv("resonance_features.csv", index=False)

Traversing directory: /content/drive/MyDrive/Data/MAS Volunteer Study March 2022
Successful files: 479
Failed files: 56
